<a href="https://colab.research.google.com/github/shalinimathi2708-cloud/shalinidiabeti/blob/main/diabetic_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded file:", zip_file)

extract_folder = "dataset"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP extracted successfully!")

Saving archive (1).zip to archive (1).zip
Uploaded file: archive (1).zip
ZIP extracted successfully!


In [ ]:
csv_file = None

for root, folders, files_list in os.walk("dataset"):
    for file in files_list:

        if file.lower().endswith(".csv"):
            csv_file = os.path.join(root, file)
            print("CSV found:", csv_file)

if csv_file is None:
    print("ERROR: No CSV file found.")

CSV found: dataset/diabetes_prediction_dataset.csv


In [ ]:
import pandas as pd

df = pd.read_csv(csv_file)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']

Missing values:
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Duplicate rows:
3854


In [ ]:
df = df.drop_duplicates()

print("Dataset after removing duplicates:")
print(df.shape)

Dataset after removing duplicates:
(96146, 9)


In [ ]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

print("Input features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Input features:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']

Target:
diabetes


In [ ]:
numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

categorical_features = [
    "gender",
    "smoking_history"
]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (76916, 8)
Testing data: (19230, 8)


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

In [ ]:
model.fit(X_train, y_train)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 96.86 %

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     17534
           1       0.94      0.69      0.80      1696

    accuracy                           0.97     19230
   macro avg       0.95      0.84      0.89     19230
weighted avg       0.97      0.97      0.97     19230



In [ ]:
probabilities = model.predict_proba(X_test)[:, 1]

print("First 10 prediction probabilities:")
print(probabilities[:10])

First 10 prediction probabilities:
[0.07  0.    0.03  0.    0.03  0.    0.    0.    0.105 0.   ]


In [ ]:
new_person = pd.DataFrame({
    "gender": ["Female"],
    "age": [25],
    "hypertension": [0],
    "heart_disease": [0],
    "smoking_history": ["never"],
    "bmi": [24.5],
    "HbA1c_level": [5.5],
    "blood_glucose_level": [110]
})

prediction = model.predict(new_person)[0]

probability = model.predict_proba(new_person)[0][1]

print("Prediction:", prediction)
print("Model probability:", round(probability * 100, 2), "%")

Prediction: 0
Model probability: 0.0 %


In [ ]:
if prediction == 1:
    print("🔴 Higher predicted diabetes risk")
else:
    print("🟢 Lower predicted diabetes risk")

🟢 Lower predicted diabetes risk


In [ ]:
import joblib

joblib.dump(model, "diabetes_model.pkl")

print("✅ Model saved as diabetes_model.pkl")

✅ Model saved as diabetes_model.pkl


In [ ]:
from google.colab import files

files.download("diabetes_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def explain_prediction(person):

    reasons = []

    glucose = person["blood_glucose_level"]
    hba1c = person["HbA1c_level"]
    bmi = person["bmi"]
    age = person["age"]
    hypertension = person["hypertension"]
    heart_disease = person["heart_disease"]

    if glucose >= 126:
        reasons.append("Blood glucose is high → increases diabetes risk")
    else:
        reasons.append("Blood glucose is below the diabetes diagnostic threshold → lower contribution")

    if hba1c >= 6.5:
        reasons.append("HbA1c is high → increases diabetes risk")
    else:
        reasons.append("HbA1c is below the diabetes diagnostic threshold → lower contribution")

    if bmi >= 25:
        reasons.append("BMI is elevated → may increase diabetes risk")
    else:
        reasons.append("BMI is in the normal range → lower contribution")

    if age >= 45:
        reasons.append("Age may contribute to increased diabetes risk")
    else:
        reasons.append("Age is relatively young → lower contribution")

    if hypertension == 1:
        reasons.append("Hypertension → increases diabetes risk")
    else:
        reasons.append("No hypertension → lower contribution")

    if heart_disease == 1:
        reasons.append("Heart disease → increases diabetes risk")
    else:
        reasons.append("No heart disease → lower contribution")

    return reasons

In [ ]:
reasons = explain_prediction(new_person.iloc[0])

for reason in reasons:
    print("•", reason)

• Blood glucose is below the diabetes diagnostic threshold → lower contribution
• HbA1c is below the diabetes diagnostic threshold → lower contribution
• BMI is in the normal range → lower contribution
• Age is relatively young → lower contribution
• No hypertension → lower contribution
• No heart disease → lower contribution


In [ ]:
!pip install pandas scikit-learn

In [ ]:
def food_assistant(person, food_habit):

    glucose = float(person["blood_glucose_level"])
    hba1c = float(person["HbA1c_level"])
    bmi = float(person["bmi"])

    advice = []

    # Blood glucose
    if glucose >= 126:
        advice.append(
            "Your blood glucose level is elevated. "
            "Try to reduce sugary drinks and highly refined carbohydrates."
        )
    else:
        advice.append(
            "Your blood glucose level is below the diabetes diagnostic threshold. "
            "Continue choosing balanced meals."
        )

    # HbA1c
    if hba1c >= 6.5:
        advice.append(
            "Your HbA1c level is elevated. "
            "Choose more fibre-rich foods such as vegetables, legumes and whole grains."
        )
    else:
        advice.append(
            "Your HbA1c level is below the diabetes diagnostic threshold. "
            "Continue maintaining balanced eating habits."
        )

    # BMI
    if bmi >= 25:
        advice.append(
            "Your BMI is elevated. "
            "Pay attention to portion sizes and include more vegetables."
        )
    else:
        advice.append(
            "Your BMI is below the overweight threshold. "
            "Continue maintaining balanced portions."
        )

    # Analyze food habit
    food = food_habit.lower()

    if "rice" in food:
        advice.append(
            "You mentioned eating rice. Consider moderating the portion "
            "and adding more vegetables, legumes and a suitable protein source."
        )

    if "sugar" in food or "sweet" in food:
        advice.append(
            "You mentioned sugar or sweets. Consider reducing foods and drinks "
            "with added sugar."
        )

    if "fried" in food:
        advice.append(
            "Try limiting frequently fried foods and choose steamed, "
            "boiled or lightly cooked foods more often."
        )

    if "fast food" in food or "junk" in food:
        advice.append(
            "Try reducing frequent fast-food or highly processed foods."
        )

    if "vegetable" not in food:
        advice.append(
            "Try including a variety of vegetables in your meals."
        )

    return advice

In [ ]:
food_habit = input(
    "🥗 Tell me about your usual food habits: "
)

print("\nFood habit entered:")
print(food_habit)

🥗 Tell me about your usual food habits: rice

Food habit entered:
rice


In [ ]:
result = food_assistant(
    new_person.iloc[0],
    food_habit
)

print("\n🤖 AI FOOD ASSISTANT")
print("=" * 50)

for i, advice in enumerate(result, start=1):
    print(f"{i}. {advice}")


🤖 AI FOOD ASSISTANT
1. Your blood glucose level is below the diabetes diagnostic threshold. Continue choosing balanced meals.
2. Your HbA1c level is below the diabetes diagnostic threshold. Continue maintaining balanced eating habits.
3. Your BMI is below the overweight threshold. Continue maintaining balanced portions.
4. You mentioned eating rice. Consider moderating the portion and adding more vegetables, legumes and a suitable protein source.
5. Try including a variety of vegetables in your meals.


In [ ]:
emergency_number = input(
    "📞 Enter your emergency contact number: "
)

print("Emergency contact saved:", emergency_number)

📞 Enter your emergency contact number: 7339353736
Emergency contact saved: 7339353736


In [ ]:
from IPython.display import display, HTML

def call_emergency_contact(phone_number):

    display(HTML(
        f'''
        <a href="tel:{phone_number}"
           style="
           display:inline-block;
           padding:12px 20px;
           background:#d32f2f;
           color:white;
           text-decoration:none;
           border-radius:8px;
           font-size:18px;">
           📞 CALL EMERGENCY CONTACT
        </a>
        '''
    ))

In [ ]:
call_emergency_contact(emergency_number)

In [ ]:
def emergency_support(user_message, emergency_number):

    message = user_message.lower()

    emergency_keywords = [
        "unconscious",
        "not responding",
        "can't breathe",
        "cannot breathe",
        "difficulty breathing",
        "chest pain",
        "severe chest pain",
        "seizure",
        "fainted",
        "severe bleeding"
    ]

    emergency_detected = any(
        keyword in message
        for keyword in emergency_keywords
    )

    if emergency_detected:

        print("\n🚨 EMERGENCY SUPPORT")
        print("=" * 50)

        print("⚠️ Possible emergency detected.")
        print("Please seek urgent medical assistance.")

        print("\n📞 Emergency contact:")
        print(emergency_number)

        call_emergency_contact(emergency_number)

    else:

        print("\n✅ No emergency keyword detected.")

In [ ]:
user_message = input(
    "🚨 Describe how you are feeling: "
)

emergency_support(
    user_message,
    emergency_number
)

🚨 Describe how you are feeling: good

✅ No emergency keyword detected.


In [ ]:
joblib.dump(model, "diabetes_model.pkl")

['diabetes_model.pkl']

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import pandas as pd
import joblib

# ============================================================
# LOAD YOUR TRAINED MODEL
# ============================================================

model = joblib.load("diabetes_model.pkl")

# ============================================================
# EMERGENCY NUMBER
# ============================================================
# Enter the trained/emergency number that you want to use.
# IMPORTANT: Replace this with your actual number.
EMERGENCY_NUMBER = "1234567890"


# ============================================================
# DIABETES PREDICTION
# ============================================================

def predict_diabetes(
    gender,
    age,
    hypertension,
    heart_disease,
    smoking_history,
    bmi,
    hba1c,
    glucose
):

    try:

        # Convert values
        age = float(age)
        hypertension = int(hypertension)
        heart_disease = int(heart_disease)
        bmi = float(bmi)
        hba1c = float(hba1c)
        glucose = float(glucose)

        # Create patient data
        person = pd.DataFrame({
            "gender": [gender],
            "age": [age],
            "hypertension": [hypertension],
            "heart_disease": [heart_disease],
            "smoking_history": [smoking_history],
            "bmi": [bmi],
            "HbA1c_level": [hba1c],
            "blood_glucose_level": [glucose]
        })

        # Prediction
        prediction = model.predict(person)[0]

        probability = model.predict_proba(person)[0][1]

        percentage = probability * 100

        # Result
        if prediction == 1:
            result = "🔴 DIABETES PREDICTED"
            message = (
                f"The model predicts a higher likelihood of diabetes.\n\n"
                f"Model probability: {percentage:.2f}%"
            )
        else:
            result = "🟢 DIABETES NOT PREDICTED"
            message = (
                f"The model predicts a lower likelihood of diabetes.\n\n"
                f"Model probability: {percentage:.2f}%"
            )

        # ====================================================
        # EXPLANATION
        # ====================================================

        reasons = []

        if glucose >= 126:
            reasons.append(
                "• Blood glucose is elevated."
            )
        else:
            reasons.append(
                "• Blood glucose is below the diabetes diagnostic threshold."
            )

        if hba1c >= 6.5:
            reasons.append(
                "• HbA1c is elevated."
            )
        else:
            reasons.append(
                "• HbA1c is below the diabetes diagnostic threshold."
            )

        if bmi >= 25:
            reasons.append(
                "• BMI is in the overweight or higher range."
            )
        else:
            reasons.append(
                "• BMI is below the overweight threshold."
            )

        if age >= 45:
            reasons.append(
                "• Age may contribute to diabetes risk."
            )
        else:
            reasons.append(
                "• Age is relatively young."
            )

        if hypertension == 1:
            reasons.append(
                "• Hypertension is present."
            )
        else:
            reasons.append(
                "• No hypertension was reported."
            )

        if heart_disease == 1:
            reasons.append(
                "• Heart disease is present."
            )
        else:
            reasons.append(
                "• No heart disease was reported."
            )

        explanation = "\n".join(reasons)

        # Store patient for food assistant
        return (
            result,
            message,
            explanation,
            person
        )

    except Exception as e:

        return (
            "❌ ERROR",
            f"Please check your input values.\n\nError: {str(e)}",
            "",
            None
        )


# ============================================================
# FOOD ASSISTANT
# ============================================================

def food_advice(food_habit, patient_data):

    if patient_data is None:
        return "⚠️ Please predict your diabetes result first."

    if food_habit is None or food_habit.strip() == "":
        return "⚠️ Please enter your usual food habits."

    try:

        person = patient_data.iloc[0]

        glucose = float(person["blood_glucose_level"])
        hba1c = float(person["HbA1c_level"])
        bmi = float(person["bmi"])

        advice = []

        advice.append("🥗 PERSONALIZED FOOD GUIDANCE")
        advice.append("=" * 40)

        # Blood glucose
        if glucose >= 126:
            advice.append(
                "• Your blood glucose is elevated. "
                "Try to reduce sugary drinks and highly refined carbohydrates."
            )
        else:
            advice.append(
                "• Your blood glucose is below the diabetes diagnostic threshold. "
                "Continue choosing balanced meals."
            )

        # HbA1c
        if hba1c >= 6.5:
            advice.append(
                "• Your HbA1c is elevated. "
                "Consider fibre-rich foods such as vegetables, legumes and whole grains."
            )
        else:
            advice.append(
                "• Your HbA1c is below the diabetes diagnostic threshold. "
                "Continue maintaining balanced eating habits."
            )

        # BMI
        if bmi >= 25:
            advice.append(
                "• Your BMI is elevated. "
                "Pay attention to portion sizes and include more vegetables."
            )
        else:
            advice.append(
                "• Your BMI is below the overweight threshold. "
                "Continue maintaining balanced portions."
            )

        # Food habit
        food = food_habit.lower()

        if "rice" in food:
            advice.append(
                "• You mentioned rice. Consider moderating the portion "
                "and adding vegetables, legumes and a suitable protein source."
            )

        if "sugar" in food or "sweet" in food:
            advice.append(
                "• You mentioned sweets or sugar. "
                "Consider reducing foods and drinks with added sugar."
            )

        if "fried" in food:
            advice.append(
                "• Try limiting frequently fried foods."
            )

        if "fast food" in food or "junk" in food:
            advice.append(
                "• Try reducing frequent fast-food or highly processed foods."
            )

        if "vegetable" not in food:
            advice.append(
                "• Try including a variety of vegetables in your meals."
            )

        advice.append("")
        advice.append(
            "⚠️ This is general educational guidance and is not a substitute "
            "for advice from a qualified healthcare professional."
        )

        return "\n".join(advice)

    except Exception as e:

        return f"❌ Error generating food advice: {str(e)}"


# ============================================================
# EMERGENCY SUPPORT
# ============================================================

def emergency_check(user_message):

    if user_message is None or user_message.strip() == "":
        return (
            "Please describe how you are feeling.",
            ""
        )

    message = user_message.lower()

    emergency_keywords = [
        "unconscious",
        "not responding",
        "can't breathe",
        "cannot breathe",
        "difficulty breathing",
        "chest pain",
        "severe chest pain",
        "seizure",
        "fainted",
        "severe bleeding"
    ]

    emergency_detected = any(
        keyword in message
        for keyword in emergency_keywords
    )

    if emergency_detected:

        result = (
            "🚨 POSSIBLE EMERGENCY DETECTED\n\n"
            "Please seek urgent medical assistance.\n\n"
            "If this is a life-threatening emergency, "
            "contact your local emergency service immediately."
        )

        # Create clickable phone link
        call_link = (
            f'<a href="tel:{EMERGENCY_NUMBER}" '
            f'style="display:inline-block;'
            f'padding:14px 25px;'
            f'background-color:#d32f2f;'
            f'color:white;'
            f'text-decoration:none;'
            f'border-radius:8px;'
            f'font-size:18px;'
            f'font-weight:bold;">'
            f'📞 CALL TRAINED CONTACT'
            f'</a>'
        )

        return result, call_link

    else:

        return (
            "✅ No emergency keyword was detected in your message.",
            ""
        )


# ============================================================
# CLEAR FUNCTION
# ============================================================

def clear_all():

    return (
        "",
        "",
        "",
        "",
        25,
        0,
        0,
        "never",
        24.5,
        5.5,
        110,
        "",
        "",
        "",
        "",
        None
    )


# ============================================================
# GRADIO FRONTEND
# ============================================================

with gr.Blocks(
    title="DiabetesCare AI",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # 🩺 DiabetesCare AI

        ### AI-based Diabetes Prediction & Food Assistance

        Enter your health details below. The system will:

        **1️⃣ Predict diabetes
        2️⃣ Explain important factors
        3️⃣ Provide food guidance
        4️⃣ Detect possible emergency messages
        5️⃣ Provide a call button for your trained emergency contact**
        """
    )

    # ========================================================
    # PATIENT DETAILS
    # ========================================================

    with gr.Tab("👤 Patient Details"):

        gr.Markdown("## Enter Your Health Information")

        with gr.Row():

            gender = gr.Dropdown(
                choices=["Female", "Male", "Other"],
                value="Female",
                label="Gender"
            )

            age = gr.Number(
                value=25,
                label="Age"
            )

        with gr.Row():

            hypertension = gr.Radio(
                choices=[0, 1],
                value=0,
                label="Hypertension (0 = No, 1 = Yes)"
            )

            heart_disease = gr.Radio(
                choices=[0, 1],
                value=0,
                label="Heart Disease (0 = No, 1 = Yes)"
            )

        smoking_history = gr.Dropdown(
            choices=[
                "never",
                "No Info",
                "current",
                "former",
                "ever",
                "not current"
            ],
            value="never",
            label="Smoking History"
        )

        with gr.Row():

            bmi = gr.Number(
                value=24.5,
                label="BMI"
            )

            hba1c = gr.Number(
                value=5.5,
                label="HbA1c Level"
            )

            glucose = gr.Number(
                value=110,
                label="Blood Glucose Level"
            )

        predict_button = gr.Button(
            "🔍 PREDICT DIABETES",
            variant="primary"
        )

    # ========================================================
    # PREDICTION RESULT
    # ========================================================

    with gr.Tab("📊 Prediction Result"):

        prediction_result = gr.Textbox(
            label="Prediction",
            lines=2
        )

        probability_result = gr.Textbox(
            label="Model Probability",
            lines=3
        )

        explanation_result = gr.Textbox(
            label="🤖 Why did the model give this result?",
            lines=10
        )

    # ========================================================
    # FOOD ASSISTANT
    # ========================================================

    with gr.Tab("🥗 AI Food Assistant"):

        gr.Markdown(
            """
            ## Tell the AI about your usual food habits

            Example:

            `I eat rice twice a day, sweets sometimes and fried food on weekends`
            """
        )

        food_input = gr.Textbox(
            label="Your Food Habits",
            placeholder="Describe your usual food habits..."
        )

        food_button = gr.Button(
            "🥗 GET FOOD GUIDANCE",
            variant="primary"
        )

        food_output = gr.Textbox(
            label="AI Food Assistant",
            lines=15
        )

    # ========================================================
    # EMERGENCY SUPPORT
    # ========================================================

    with gr.Tab("🚨 Emergency Support"):

        gr.Markdown(
            """
            ## 🚨 Emergency Support

            Describe your current situation.

            Example:

            `I have severe chest pain and difficulty breathing`
            """
        )

        emergency_input = gr.Textbox(
            label="Describe Your Situation",
            placeholder="Describe how you are feeling..."
        )

        emergency_button = gr.Button(
            "🚨 CHECK EMERGENCY",
            variant="stop"
        )

        emergency_output = gr.Textbox(
            label="Emergency Status",
            lines=6
        )

        call_button = gr.HTML(
            label="Emergency Contact"
        )

        gr.Markdown(
            f"""
            **Trained Emergency Contact:** `{EMERGENCY_NUMBER}`

            ⚠️ Replace the number in the code with your intended trained
            contact number before using this feature.
            """
        )

    # ========================================================
    # CLEAR BUTTON
    # ========================================================

    clear_button = gr.Button(
        "🔄 CLEAR ALL"
    )

    # Hidden state for patient information
    patient_state = gr.State(None)

    # ========================================================
    # BUTTON ACTIONS
    # ========================================================

    predict_button.click(
        fn=predict_diabetes,
        inputs=[
            gender,
            age,
            hypertension,
            heart_disease,
            smoking_history,
            bmi,
            hba1c,
            glucose
        ],
        outputs=[
            prediction_result,
            probability_result,
            explanation_result,
            patient_state
        ]
    )

    food_button.click(
        fn=food_advice,
        inputs=[
            food_input,
            patient_state
        ],
        outputs=food_output
    )

    emergency_button.click(
        fn=emergency_check,
        inputs=emergency_input,
        outputs=[
            emergency_output,
            call_button
        ]
    )

    clear_button.click(
        fn=clear_all,
        inputs=[],
        outputs=[
            prediction_result,
            probability_result,
            explanation_result,
            food_input,
            age,
            hypertension,
            heart_disease,
            smoking_history,
            bmi,
            hba1c,
            glucose,
            food_output,
            emergency_input,
            emergency_output,
            call_button,
            patient_state
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_2395/1964925865.py:357: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a8f12f0e3bacaffe8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://23166bb3e731583fcb.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://2c37cc85602d4ec900.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://ad38c9d222fb382e0b.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://a8f12f0e3bacaffe8a.gradio.live
